In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "frcrn-vlsp"
RUN_TRAINING = False
RESUME_RUN_ID = None

OUTPUT_DIR = ROOT / "output" if RUN_TRAINING else ROOT / "output" / "test"
run = (
    RunPaths.from_run_id(RESUME_RUN_ID, OUTPUT_DIR) if RESUME_RUN_ID else RunPaths.new(PIPELINE_NAME, OUTPUT_DIR)
).create()

logger = setup_logger(run.run_id, run.log_file)
logger.info("pipeline=%s | run=%s | mode=%s", PIPELINE_NAME, run.run_id, "TRAINING" if RUN_TRAINING else "SMOKE TEST")
logger.info(
    "log=%s | checkpoint=%s | result=%s | figure=%s", run.log_file, run.checkpoint_dir, run.result_dir, run.figure_dir
)

In [ ]:
import os
from dataclasses import replace

import torch
import yaml

from src.train import FRCRNTrainConfig, scale_batch_size
from src.utils.seed import set_seed

with open(ROOT / "config" / "beat.yaml", encoding="utf-8") as _f:
    beat_yaml = yaml.safe_load(_f)
with open(ROOT / "config" / "frcrn.yaml", encoding="utf-8") as _f:
    frcrn_yaml = yaml.safe_load(_f)
with open(ROOT / "config" / "vlsp.yaml", encoding="utf-8") as _f:
    vlsp_yaml = yaml.safe_load(_f)

ALL_SNR = (-5, 0, 5, 10, 15, 20)

CONFIG = {
    "seed": beat_yaml["train"]["seed"],
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers": min(os.cpu_count() or 4, 8),
    "resume": RESUME_RUN_ID is not None,
    "max_samples": None,
    "drive_upload": RUN_TRAINING,
    "sample_rate": beat_yaml["frontend"]["sample_rate"],
    "snr_groups": {
        "low_snr": (-5, 0),
        "mid_snr": (5, 10),
        "high_snr": (15, 20),
        "all_snr": ALL_SNR,
    },
    "frcrn": FRCRNTrainConfig.from_yaml(ROOT / "config" / "frcrn.yaml"),
}

if not RUN_TRAINING:
    CONFIG.update(max_samples=8, num_workers=0)
    CONFIG["frcrn"] = replace(
        CONFIG["frcrn"], epochs=1, batch_size=4, eval_batch_size=4, segment_seconds=1.0, eval_segment_seconds=1.0
    )

DEVICE = torch.device(CONFIG["device"])
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
if RUN_TRAINING and DEVICE.type == "cuda":
    CONFIG["frcrn"] = scale_batch_size(
        CONFIG["frcrn"], ROOT / "config" / "frcrn.yaml", DEVICE, CONFIG["sample_rate"], logger
    )
set_seed(CONFIG["seed"])
logger.info(
    "run_training=%s | device=%s | seed=%d | resume=%s | max_samples=%s",
    RUN_TRAINING,
    DEVICE,
    CONFIG["seed"],
    CONFIG["resume"],
    CONFIG["max_samples"],
)
logger.info(
    "snr_groups=%s | frcrn epochs=%d patience=%d batch=%d lr=%.1e | vlsp split=%s",
    CONFIG["snr_groups"],
    CONFIG["frcrn"].epochs,
    CONFIG["frcrn"].patience,
    CONFIG["frcrn"].batch_size,
    CONFIG["frcrn"].lr,
    vlsp_yaml["split"],
)

In [ ]:
from scripts.download_dataset import ensure_dataset
from scripts.load_dataset import (
    AUDIOSET_KEY,
    MIXTURE_KEY,
    VLSP_KEY,
    AudioSet40Dataset,
    VLSPMixDataset,
    build_dataloader,
    read_noise_ytids,
    subsample,
)

for _key in (AUDIOSET_KEY, MIXTURE_KEY, VLSP_KEY):
    ensure_dataset(_key, ROOT / "datasets")

held_out_noise = read_noise_ytids(splits=("validation", "test"))
frcrn_config = CONFIG["frcrn"]
max_samples = CONFIG["max_samples"]


def make_loader(dataset, batch_size, shuffle):
    return build_dataloader(dataset, batch_size, shuffle, CONFIG["num_workers"])


def make_mix(split, noise_split, snr_levels, duration, deterministic, seed=0):
    return VLSPMixDataset(
        split=split,
        noise_dataset=AudioSet40Dataset(split=noise_split, duration=None, exclude_ytids=held_out_noise),
        snr_levels=snr_levels,
        duration=duration,
        deterministic=deterministic,
        seed=seed,
    )


vlsp_train = make_mix("train", "train", frcrn_config.snr_levels, frcrn_config.segment_seconds, False)
vlsp_val = make_mix(
    "validation", "validation", frcrn_config.snr_levels, frcrn_config.eval_segment_seconds, True, CONFIG["seed"]
)
frcrn_loaders = {
    "train": make_loader(subsample(vlsp_train, max_samples), frcrn_config.batch_size, True),
    "validation": make_loader(subsample(vlsp_val, max_samples), frcrn_config.eval_batch_size, False),
}


def make_test_loader(snr_levels):
    dataset = make_mix("test", "validation", snr_levels, frcrn_config.eval_segment_seconds, True, CONFIG["seed"] + 1)
    return make_loader(subsample(dataset, max_samples), frcrn_config.eval_batch_size, False)


logger.info(
    "%s (frcrn): train speech=%d noise=%d | val speech=%d noise=%d | excluded noise ytids=%d",
    VLSP_KEY,
    len(vlsp_train),
    len(vlsp_train.noise_dataset),
    len(vlsp_val),
    len(vlsp_val.noise_dataset),
    len(held_out_noise),
)
for _group, _levels in CONFIG["snr_groups"].items():
    logger.info("%s (frcrn) test snr=%s: %d samples", VLSP_KEY, _levels, len(make_test_loader(_levels).dataset))

In [ ]:
from scripts.load_pretrain import PRETRAINED, download_pretrained

logger.info("downloading / locating pretrained checkpoint")
frcrn_checkpoint = download_pretrained("frcrn")
logger.info("frcrn pretrain: %s (%.1f MB)", frcrn_checkpoint.name, frcrn_checkpoint.stat().st_size / 1e6)

In [ ]:
frcrn_spec = PRETRAINED["frcrn"]

separator = frcrn_spec.model_cls.from_yaml(frcrn_spec.config_path)
frcrn_report = separator.load_pretrained(frcrn_checkpoint)
logger.info(
    "frcrn: pretrained loaded | missing=%d unexpected=%d | target=%s",
    len(frcrn_report.missing),
    len(frcrn_report.unexpected),
    separator.config.target,
)
logger.info("params: frcrn=%.2fM", separator.num_parameters() / 1e6)

In [ ]:
import csv
import json

from scripts.drive_sync import DRIVE_FRCRN_VLSP_DIR, DriveCheckpointUploader, DriveLogUploader
from src.train import FRCRNTrainer
from src.utils.checkpoint import load_checkpoint

frcrn_ckpt_path = run.checkpoint_path("frcrn")
frcrn_result_path = run.history_path("frcrn_finetune")
frcrn_test_path = run.history_path("frcrn_test")
metric_names = ["noise_si_snr", "loss"]

drive_uploader = None
log_uploader = None
if CONFIG["drive_upload"]:
    from gdrive_fsspec import GoogleDriveFileSystem

    drive_fs = GoogleDriveFileSystem(
        use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False}
    )
    drive_fs.makedirs(DRIVE_FRCRN_VLSP_DIR, exist_ok=True)
    drive_uploader = DriveCheckpointUploader(drive_fs, DRIVE_FRCRN_VLSP_DIR, logger)
    logger.info("drive upload: best.pt -> %s", drive_uploader.remote_path)
    log_uploader = DriveLogUploader(drive_fs, run.log_file, DRIVE_FRCRN_VLSP_DIR)
    logger.addHandler(log_uploader)
    logger.info("drive upload: log -> %s", log_uploader.remote_path)

frcrn_trainer = FRCRNTrainer(
    separator, frcrn_config, DEVICE, logger, "frcrn", frcrn_ckpt_path, on_checkpoint=drive_uploader
)
separator.to(DEVICE)


def evaluate_groups(stage):
    rows = {}
    for group, levels in CONFIG["snr_groups"].items():
        name = f"{group}_{stage}"
        test = frcrn_trainer.evaluate(make_test_loader(levels), desc=f"{name} - test")
        rows[name] = {"scenario": name, "snr_group": group, "snr_levels": list(levels), "stage": stage, "test": test}
        logger.info("%s: test noise si-snr %.2f dB | loss %.3f", name, test["noise_si_snr"], test["loss"])
    return rows


def write_result_csv(rows, fit):
    columns = ["scenario", "snr_group", "snr_levels", "stage", "best_epoch", "epochs_run", *metric_names]
    with open(run.result_file, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore", restval="")
        writer.writeheader()
        for row in rows.values():
            extra = (
                {"best_epoch": fit["best_epoch"], "epochs_run": fit["epochs_run"]}
                if row["stage"] == "finetuned"
                else {}
            )
            writer.writerow({**row, "snr_levels": "|".join(str(s) for s in row["snr_levels"]), **row["test"], **extra})


logger.info("stage 1/3: evaluate pretrained frcrn on %s test", VLSP_KEY)
test_results = evaluate_groups("pretrained")

logger.info("stage 2/3: fine-tune frcrn on %s", VLSP_KEY)
if CONFIG["resume"] and frcrn_ckpt_path.is_file():
    load_checkpoint(frcrn_ckpt_path, separator.network)
    fit_summary = json.loads(frcrn_result_path.read_text(encoding="utf-8"))
    logger.info("frcrn: resumed from %s", frcrn_ckpt_path.name)
else:
    frcrn_result = frcrn_trainer.fit(frcrn_loaders["train"], frcrn_loaders["validation"])
    fit_summary = frcrn_result.to_dict()
    frcrn_result_path.write_text(json.dumps(fit_summary, indent=2), encoding="utf-8")
    logger.info(
        "frcrn: validation noise si-snr %.2f -> %.2f dB (best epoch %d)",
        frcrn_result.extras["baseline_score"],
        frcrn_result.best_score,
        frcrn_result.best_epoch,
    )

logger.info("stage 3/3: evaluate fine-tuned frcrn on %s test", VLSP_KEY)
test_results.update(evaluate_groups("finetuned"))
frcrn_test_path.write_text(json.dumps(test_results, indent=2), encoding="utf-8")
write_result_csv(test_results, fit_summary)

header = f"{'scenario':<22}" + "".join(f"{m:>16}" for m in metric_names)
print(header)
print("-" * len(header))
for _row in test_results.values():
    print(f"{_row['scenario']:<22}" + "".join(f"{_row['test'][m]:>16.4f}" for m in metric_names))
logger.info("result saved to %s", run.result_file)
if log_uploader is not None:
    log_uploader.upload()

In [ ]:
def show_table(title, rows):
    width = max(len(str(key)) for key, _ in rows) + 2
    print(f"\n{title}\n" + "=" * (width + 40))
    for key, value in rows:
        print(f"{key!s:<{width}}{value}")


def millions(module):
    return sum(p.numel() for p in module.parameters()) / 1e6


frcrn_model_config = separator.config
with torch.no_grad():
    dummy = torch.zeros(1, int(frcrn_config.eval_segment_seconds * CONFIG["sample_rate"]))
    noise_estimate = separator.cpu()(dummy)

show_table(
    "Pipeline",
    [
        ("fine-tune", "VLSP speech + AudioSet-40l noise at random SNR -> FRCRN -> noise estimate"),
        ("input waveform", tuple(dummy.shape)),
        ("FRCRN noise estimate", tuple(noise_estimate.shape)),
        ("selection metric", "validation noise_si_snr"),
    ],
)

show_table(
    "FRCRN architecture",
    [
        *frcrn_model_config.to_dict().items(),
        ("stft bins", frcrn_model_config.num_bins),
        ("cascaded UNets", 2),
        ("encoder / decoder blocks per UNet", len(separator.network.unet.encoders)),
        ("channels", 128),
        ("FSMN order", 20),
        ("mask", "cIRM, tanh(u1) + tanh(u2(u1))"),
        ("parameters (M)", f"{millions(separator):.2f}"),
    ],
)

show_table("FRCRN fine-tune hyperparameters", list(frcrn_config.to_dict().items()))
show_table("Loss", [("FRCRN", f"cIRM MSE x fft_len - SI-SNR | {frcrn_yaml['loss']}")])
show_table(
    "Dataset",
    [
        ("speech", f"{VLSP_KEY}, split by path hash {vlsp_yaml['split']}"),
        ("noise", f"{AUDIOSET_KEY}, noise ytids used by {MIXTURE_KEY} validation/test excluded"),
        ("test noise pool", "AudioSet-40l validation split (no test split in AudioSet-40l)"),
        ("train / validation / test speech", (len(vlsp_train), len(vlsp_val), len(make_test_loader(ALL_SNR).dataset))),
    ],
)
show_table(
    "Experiment grid",
    [
        ("snr groups", CONFIG["snr_groups"]),
        ("stages", ("pretrained", "finetuned")),
        ("rows in result.csv", len(CONFIG["snr_groups"]) * 2),
        ("outputs", f"{run.log_file} | {run.checkpoint_dir} | {run.result_dir}"),
    ],
)

In [ ]:
from IPython.display import Image, display

from src.utils.plots import plot_frcrn_curve, plot_frcrn_test

figures = [
    plot_frcrn_curve(json.loads(frcrn_result_path.read_text(encoding="utf-8")), run.figure_dir / "frcrn_si_snr.png"),
    plot_frcrn_test(test_results, list(CONFIG["snr_groups"]), run.figure_dir / "frcrn_test.png"),
]
for _figure in figures:
    logger.info("figure saved: %s", _figure)
    display(Image(filename=str(_figure)))